**Lane:** CTR / Engagement Opportunity Scoring

**ML Task Type:** Ranking with regression

**Why ranking?**
The output is a ranked list of pages sorted by opportunity. A content team can only review 20-50 pages per week, so we need the best candidates at the top.

**Why regression?**
To find opportunities, we first need to estimate expected CTR for each page based on its position and other features. This is a regression problem.

**The ML loop:**
1. Input: Page features (position, impressions, CTR, content age, etc.)
2. Model: Predicts expected CTR for each page
3. Calculate gap: Expected CTR - Actual CTR
4. Rank pages by gap size
5. Output: Ranked list of pages with recommendations

**Target:** CTR gap (expected CTR - actual CTR)

**Why this target?**
- It directly measures opportunity - pages with big gaps could improve with changes
- It accounts for position - a page in position 1 should get more clicks than position 5
- It's business-relevant - improving CTR means more traffic

**What the target column would look like:**
A continuous value from -0.10 to +0.10, where positive means underperforming.

**Primary Metric:** Precision@K (specifically Precision@20 and Precision@50)

**What this means:**
- Of the top 20 pages we flag, how many actually have a meaningful CTR gap?
- Of the top 50 pages, how many actually have a meaningful CTR gap?

**Why this metric:**
- The output is a ranking - we need to know if we're putting the right pages at the top
- It's business-relevant: wasted time on false positives is expensive

**Secondary metrics:**
- Average Precision (AP): How good is the overall ranking?
- Gap size distribution: Are we finding meaningful opportunities or small gaps?

In [1]:
import pandas as pd

# Load data
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Show unit of analysis: one row = one page
print("Unit of analysis: one row = one page")
print(f"Total pages: {len(df)}")
print("\nSample of the data:")
df[["content_id", "position_tier", "impressions_90d", "ctr", "trend_direction"]].head(10)

Unit of analysis: one row = one page
Total pages: 30000

Sample of the data:


,content_id,position_tier,impressions_90d,ctr,trend_direction
0,content_304f48230142,striking,3803,0.76,down
1,content_a1fb4e703a9e,page_3_5,15320,0.05,down
2,content_9aa793d4d895,page_3_5,12581,0.09,down
3,content_331d6c4de07b,page_1,11751,0.49,stable
4,content_d99b7a2d90ca,page_3_5,19140,0.13,down
5,content_d4084a4bc775,page_1,3970,0.03,down
6,content_9a34b442b552,page_1,20,0.00,down
7,content_a63219c6e95a,page_3_5,1724,0.06,stable
8,content_5e6c160719bc,page_3_5,32574,0.09,down
9,content_c27558df2b0c,page_1,1240,0.16,down


A fixed rule like "flag CTR < 5%" would miss the context of position. A page in position 10 with 4% CTR is doing fine, while a page in position 1 with 6% CTR is underperforming. ML solves this by learning the relationship between position and expected CTR, then flagging pages based on the gap rather than a fixed threshold.

1. Is my task type clearly defined?
   Yes - ranking with regression.

2. Is my target measurable?
   Yes - CTR gap is a continuous value we can calculate.

3. Is my success metric tied to a real decision?
   Yes - Precision@K tells us if the top pages are worth reviewing.

4. Is my unit of analysis appropriate?
   Yes - one page per row matches the decision grain.

5. Is ML actually better than a fixed rule here?
   Yes - a fixed rule ignores position and context.